# Orbit Wars: Spatial U-Net Training

## 1. Setup Your Kaggle Environment

Before running this notebook, click **Add Data** in the top right and add two things:
1. **Your Code**: Upload your zipped `OrbitsWar` codebase as a Kaggle dataset.
2. **The Chunks**: Zip the `imitation/chunks` folder from your PC and upload it as a second Kaggle dataset.

In [ ]:
import os
import sys
import glob
import shutil

# Find where Kaggle mounted your codebase
found_scripts = glob.glob('/kaggle/input/**/train.py', recursive=True)
if found_scripts:
    code_dir = os.path.dirname(found_scripts[0])
    sys.path.insert(0, code_dir)
    print(f"Found training code at: {code_dir}")
else:
    print("ERROR: Could not find train.py in /kaggle/input/")

## 2. Link the Dataset Chunks

We need to copy the `.npz` chunks into the working directory so `train.py` can stream them.

In [ ]:
# Find where Kaggle mounted your chunks dataset
found_chunks = glob.glob('/kaggle/input/**/dataset_chunk_0000.npz', recursive=True)
if found_chunks:
    chunks_dir = os.path.dirname(found_chunks[0])
    print(f"Found chunks dataset at: {chunks_dir}")
    
    # Create chunks folder in working directory
    working_chunks = '/kaggle/working/imitation/chunks'
    os.makedirs(working_chunks, exist_ok=True)
    
    # Symlink all chunks to trick train.py into finding them
    for chunk in glob.glob(f"{chunks_dir}/*.npz"):
        dest = os.path.join(working_chunks, os.path.basename(chunk))
        if not os.path.exists(dest):
            os.symlink(chunk, dest)
    print("Successfully linked all chunks to /kaggle/working/imitation/chunks!")
else:
    print("ERROR: Could not find dataset_chunk_0000.npz in /kaggle/input/")

## 3. Train the U-Net Matrix Model

Make sure your Notebook settings (right sidebar) have **Accelerator: GPU T4 x2** turned on!

In [ ]:
import train
# Train the Spatial U-Net for 10 epochs
train.train(epochs=10, batch_size=128)